# v0.17.0 — Model-level authentication (`AuthenticatedUserMixin`)

Where v0.16.0 authenticates the **connection**, v0.17.0 authenticates a **model**: declare a
user model, let the ORM generate its `DEFINE ACCESS` statement, and get **hydrated model
instances** back instead of bare tokens.

| Entry point | What it does |
| ----------- | ------------ |
| `access_ddl()` | render the DDL — pure, touches no database |
| `define_access()` | apply it; idempotent, safe at start-up |
| `signup(**fields)` | register a record user → `AuthResult[Model]` |
| `signin(**credentials)` | authenticate one → `AuthResult[Model]` |
| `authenticate(token)` | stored JWT → model instance |
| `refresh(token)` | renew a session — **SurrealDB 3.x only** |

**The headline is session isolation.** Every call runs on its own short-lived connection, so
the process-wide client keeps the identity `set_connection()` gave it. Section 7 proves it.

Everything here behaves **identically on SurrealDB 2.6.x and 3.x** except `with_refresh=True`
and `refresh()`: 2.6.x cannot parse `WITH REFRESH`. Section 8 probes for that and explains the
fallback rather than failing.


## 1. Connect

The URL is env-overridable so you can point this at your own server.


In [1]:
import os
from uuid import uuid4

from surreal_orm_lite import (
    AuthenticatedUserMixin,
    BaseSurrealModel,
    SurrealConfigDict,
    SurrealDBConnectionManager,
    SurrealDbAuthenticationError,
    SurrealDbError,
)

HOST = os.environ.get("SURREALDB_HOST", "localhost")
PORT = os.environ.get("SURREALDB_PORT", "8000")
SurrealDBConnectionManager.set_connection(
    url=f"ws://{HOST}:{PORT}/rpc",
    user="root", password="root",
    namespace="examples", database="examples",
)
print("Connection configured:", SurrealDBConnectionManager.is_connection_set())


Connection configured: True


## 2. Declare the user model

Mix `AuthenticatedUserMixin` in **before** `BaseSurrealModel` so its classmethods take
precedence.

The model must declare either an `id` field or a `primary_key` — the ORM's standing contract for
every model. Prefer `id` for auth models: SIGNUP lets the server mint the record id, and this is
the field that receives it.


In [2]:
class AppUser(AuthenticatedUserMixin, BaseSurrealModel):
    model_config = SurrealConfigDict(
        access_name="account",
        identifier_field="email",
        password_field="password",
    )

    id: str | None = None
    email: str
    password: str
    name: str = ""


print("access name:      ", AppUser.get_access_name())
print("identifier field: ", AppUser.get_identifier_field())
print("password field:   ", AppUser.get_password_field())


access name:       account
identifier field:  email
password field:    password


## 3. Inspect the DDL before applying it

`access_ddl()` is **pure** — it renders the SurrealQL and returns it without touching the
database, so you can print it, diff it, or feed it to a migration.

Note the `DEFINE TABLE`: it ships **by default**. Without `FOR select WHERE id = $auth.id` a
signin succeeds and `$auth` is set, yet the server returns no record — so no instance could be
built. Section 6 demonstrates that on purpose.


In [3]:
for statement in AppUser.access_ddl():
    print(statement)
    print()


DEFINE TABLE OVERWRITE AppUser SCHEMALESS PERMISSIONS FOR select, update WHERE id = $auth.id;

DEFINE ACCESS OVERWRITE account ON DATABASE TYPE RECORD SIGNUP ( CREATE AppUser SET email = $email, password = crypto::argon2::generate($password), name = $name ) SIGNIN ( SELECT * FROM AppUser WHERE email = $email AND crypto::argon2::compare(password, $password) ) DURATION FOR TOKEN 15m, FOR SESSION 12h;



## 4. Apply it

`define_access()` is idempotent with the default `overwrite=True`, so it is safe to call on
every application start-up. It returns the statements it applied.


In [4]:
applied = await AppUser.define_access()
print(f"applied {len(applied)} statements")

# Idempotent: running it again just converges the database onto the model.
await AppUser.define_access()
print("second run: no error")


applied 2 statements
second run: no error


## 5. Sign up and sign in

Both return an `AuthResult`, pairing the hydrated model instance with its tokens.


In [5]:
email = f"{uuid4().hex}@example.test"

result = await AppUser.signup(email=email, password="s3cret-passphrase", name="Ada")

print("type: ", type(result.user).__name__)
print("email:", result.user.email)
print("name: ", result.user.name)
print("id:   ", result.user.get_id())
print("has an access token:", bool(result.tokens.access))


type:  AppUser
email: 6d7e2c1c78384d198b1fdf32fca5c7f6@example.test
name:  Ada
id:    6wjolxb8ke2rgh1ta2io
has an access token: True


`AuthResult`'s repr **redacts** — it delegates to `AuthTokens`, so a JWT never reaches a log
line, a traceback or an assertion diff. Read `result.tokens.access` explicitly when you need it.


In [6]:
print(repr(result.tokens))


AuthTokens(access=<redacted>, refresh=None)


The password field carries the **hash**, never the plaintext you submitted: SurrealDB returns
the stored record verbatim, and hiding that would be a worse surprise than showing it.


In [7]:
print("submitted: s3cret-passphrase")
print("stored:   ", result.user.password[:32], "…")


submitted: s3cret-passphrase
stored:    $argon2id$v=19$m=19456,t=2,p=1$Q …


Signing in returns the same record.


In [8]:
signed_in = await AppUser.signin(email=email, password="s3cret-passphrase")

print("same record:", signed_in.user.get_id() == result.user.get_id())
print("name:       ", signed_in.user.name)


same record: True
name:        Ada


A wrong password raises `SurrealDbAuthenticationError` on **both** DB lines. The raw SDK error
differs (`NotFoundError` on 3.x, `InternalError` on 2.6.x); the ORM normalises it.


In [9]:
try:
    await AppUser.signin(email=email, password="wrong")
except SurrealDbAuthenticationError as exc:
    print("refused:", str(exc)[:90], "…")


refused: Authentication failed during signin: No record was returned …


### `authenticate()` — the per-request half of a web login

Hand the stored JWT back and get the current user.


In [10]:
me = await AppUser.authenticate(result.tokens.access)

print("type: ", type(me).__name__)
print("email:", me.email)
print("same record:", me.get_id() == result.user.get_id())


type:  AppUser
email: 6d7e2c1c78384d198b1fdf32fca5c7f6@example.test
same record: True


## 6. When the record cannot read itself

If the table denies the record `select` on itself, signin *succeeds* but the server returns no
record — v0.16.0's quietest trap. A model-level API promises an instance, so instead of handing
back `None` it raises an error that **names the missing permission**.


In [11]:
client = await SurrealDBConnectionManager.get_client()
await client.query("DEFINE TABLE OVERWRITE AppUser SCHEMALESS PERMISSIONS NONE;", {})
await AppUser.define_access(with_table=False)   # keep the restrictive table

try:
    await AppUser.signup(email=f"{uuid4().hex}@example.test", password="s3cret-passphrase")
except SurrealDbAuthenticationError as exc:
    print(exc)

# Put the working permissions back.
await AppUser.define_access()
print("\npermissions restored")


AppUser.signup() authenticated successfully but the server returned no record, so no instance can be built. The usual cause is table PERMISSIONS: the record cannot select itself. Grant it with 'DEFINE TABLE AppUser PERMISSIONS FOR select WHERE id = $auth.id;' or let define_access() do it (with_table=True, the default).

permissions restored


## 7. Session isolation — the point of v0.17.0

v0.16.0's connection-level `signin()` re-identifies the client **every model shares**, which
makes it unusable per-request in a concurrent server. v0.17.0 runs each call on its own
short-lived connection instead.

Below: after signing a record user in, the shared connection is still root — it can still run
`INFO FOR DB;`, which a record user never could.


In [12]:
before = SurrealDBConnectionManager.get_session_token()

await AppUser.signin(email=email, password="s3cret-passphrase")

after = SurrealDBConnectionManager.get_session_token()
print("shared session token unchanged:", before == after)

# A record user could never run INFO FOR DB. Deriving the answer from the call itself,
# rather than printing a hopeful literal.
client = await SurrealDBConnectionManager.get_client()
try:
    await client.query("INFO FOR DB;", {})
    still_root = True
except Exception:
    still_root = False
print("shared connection still has root powers:", still_root)


shared session token unchanged: True
shared connection still has root powers: True


Pass `bind=True` when you *do* want the process-wide session to become that user — convenient
in a script or a notebook, wrong in a server handling concurrent users.


In [13]:
bound = await AppUser.signin(email=email, password="s3cret-passphrase", bind=True)

print("shared token is now the record user's:",
      SurrealDBConnectionManager.get_session_token() == bound.tokens.access)

# Back to the configured root identity.
await SurrealDBConnectionManager.invalidate()
print("after invalidate(), shared token:", SurrealDBConnectionManager.get_session_token())


shared token is now the record user's: True
after invalidate(), shared token: None


## 8. Refresh tokens — SurrealDB 3.x only

`with_refresh=True` emits `WITH REFRESH`, which **SurrealDB 2.6.x cannot parse at all**. Rather
than failing there, this section probes by attempting the DDL and explains the fallback — the
notebook's version of the test suite's `pytest.skip()`.


In [14]:
class RefreshUser(AuthenticatedUserMixin, BaseSurrealModel):
    model_config = SurrealConfigDict(access_name="refresh_account", with_refresh=True)

    id: str | None = None
    email: str
    password: str


try:
    await RefreshUser.define_access()
    refresh_supported = True
except SurrealDbError as exc:
    refresh_supported = False
    print("This server does not support WITH REFRESH (SurrealDB 2.6.x).")
    print("The ORM says so explicitly rather than leaking a parse error:\n")
    print(str(exc)[-180:])

print("\nrefresh supported:", refresh_supported)



refresh supported: True


On 3.x, signup now also yields a **refresh token**, and exchanging it returns a renewed session.

> ⚠️ **Refresh tokens rotate.** A successful exchange kills the token it spent, immediately and
> permanently. Persist the new one or the user is logged out for good — with nothing raised at
> the moment you make the mistake.


In [15]:
if refresh_supported:
    r_email = f"{uuid4().hex}@example.test"
    created = await RefreshUser.signup(email=r_email, password="s3cret-passphrase")
    print("refresh token issued:", created.tokens.refresh is not None)

    renewed = await RefreshUser.refresh(created.tokens.refresh)
    print("renewed the same record:", renewed.user.get_id() == created.user.get_id())

    # Rotation: the spent token is dead.
    try:
        await RefreshUser.refresh(created.tokens.refresh)
    except SurrealDbAuthenticationError:
        print("the spent refresh token was rejected: True")
else:
    print("Skipped: this server has no refresh tokens.")
    print("On 2.6.x, AuthResult.tokens.refresh is always None — use signin() with the")
    print("password to start a new session instead.")


refresh token issued: True
renewed the same record: True
the spent refresh token was rejected: True


## 9. Cleanup


In [16]:
client = await SurrealDBConnectionManager.get_client()

for statement in (
    "REMOVE ACCESS account ON DATABASE;",
    "REMOVE ACCESS refresh_account ON DATABASE;",
    "REMOVE TABLE AppUser;",
    "REMOVE TABLE RefreshUser;",
):
    try:
        await client.query(statement, {})
    except Exception:
        pass  # already gone (e.g. the 2.6.x refresh access was never created)

await SurrealDBConnectionManager.close_connection()
print("cleaned up")


cleaned up


---

## Summary

| | 2.6.x | 3.x |
| --- | --- | --- |
| `access_ddl()` / `define_access()` | same | same |
| `signup()` / `signin()` / `authenticate()` | same | same |
| Session isolation (`bind=False` default) | same | same |
| `with_refresh=True` | raises, message names the requirement | supported |
| `refresh()` | unavailable | supported, rotating |

Model-level auth is the model-shaped counterpart to v0.16.0's connection-level auth — reach for
`SurrealDBConnectionManager.signin()` when the *process* has one identity, and for
`Model.signin()` when each request has its own.
